# 10 — Rock J-space: smoke, then 100 responses × 5 positions

**Question.** Does sparse nonnegative decomposition in a J-Lens dictionary
recover Rock-specific information that ordinary vocabulary ranking misses?

This is a Rock-adapter-only experiment at the manually fixed source layer 40.
It never generates new answers: it replays the exact 100 saved standard TEST
responses from notebook 07 and records five pre-registered response positions.

The notebook has two hard stages:

1. a two-response end-to-end smoke with dictionary/logit and
   TransformerLens-parity gates;
2. a resumable 100 × 5 full sweep, which remains disabled until the smoke passes.

Long model work is implemented in `scripts/run_rock_jspace.py` so it can run
atomically under `tmux`; the key dictionary and pursuit functions remain
directly inspectable below.

## Frozen design

- **Adapter condition:** Rock LoRA only.
- **Examples:** the 100 saved `standard/test/rock` responses; literal leaks are
  retained in raw artifacts and excluded from headline analysis in notebook 11.
- **Layer:** 40 only; no layer search.
- **Positions:** first, 25%, middle, 75%, and last generated token.
- **Primary sparse method:** full-vocabulary Gradient Pursuit, `k=16`.
- **Leakage control:** every token ID emitted anywhere in that response is
  unavailable to the primary sparse support.
- **Dictionaries:** public base-model J-Lens n=1000 and Rock-specific J-Lens n=100.
- **Ordinary baselines:** Logit Lens, public J-Lens, Rock-specific J-Lens.
- **Robustness subset:** first 10 prompts with GP `k=8/25` and unmasked NNOMP
  `k=16` clearly labelled as a diagnostic, not a headline comparison.

The same Rock activations feed every method. J-space demonstrates decodability
under this decomposition; it does not show causal use by the model.

In [ ]:
from __future__ import annotations

import inspect
import json
import os
import shlex
import subprocess
import sys
import time
from importlib.metadata import version
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.experiment_io import load_json
from src.jspace import (
    build_effective_jlens_dictionary,
    masked_gradient_pursuit,
    response_anchor_indices,
)

CONFIG_PATH = PROJECT_ROOT / "configs" / "rock_jspace.json"
config = load_json(CONFIG_PATH)
PYTHON = sys.executable
PROJECT_ROOT

## Inspect the two critical operations

Qwen applies a learned final RMSNorm before the language-model head. The
stored RMS parameter is a delta, so the dictionary uses
`(W_U[token] * (1 + rms_weight)) @ J`; the per-activation
RMS denominator is positive and does not change vocabulary rank.

`masked_gradient_pursuit` follows TransformerLens 3.8.1 and adds only an
excluded-token mask. The smoke requires exact unmasked support parity and
close coefficients against TransformerLens itself.

In [ ]:
print(inspect.getsource(build_effective_jlens_dictionary))
print(inspect.getsource(masked_gradient_pursuit))

## Dependency and artifact preflight

This cell is read-only. It reports `waiting_for_rock_n100` while notebook 09's
standalone refit is still running. Do not launch the smoke until it reports
`ready`; the runner also refuses to load another 27B model unless at least
70 GiB of GPU memory is free.

In [ ]:
expected_transformer_lens = config["jspace"]["transformer_lens_version"]
try:
    actual_transformer_lens = version("transformer-lens")
except Exception:
    actual_transformer_lens = None
print({
    "expected_transformer_lens": expected_transformer_lens,
    "actual_transformer_lens": actual_transformer_lens,
})
preflight = subprocess.run(
    [PYTHON, "scripts/run_rock_jspace.py", "preflight"],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
)
print(preflight.stdout)
if preflight.stderr:
    print(preflight.stderr)
print("preflight return code:", preflight.returncode)

## Cheap solver tests

These tests use tiny synthetic dictionaries and no model. They verify named
anchor selection, nonnegative reconstruction, and that excluded atoms cannot
enter the support. They are useful before the H100 is free, but they do not
replace the real full-vocabulary parity smoke.

In [ ]:
solver_tests = subprocess.run(
    [PYTHON, "scripts/check_jspace_solver.py"],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
)
print(solver_tests.stdout)
if solver_tests.stderr:
    print(solver_tests.stderr)
assert solver_tests.returncode == 0

## Stage A — two-response GPU smoke

The smoke loads pinned Qwen BF16 + Rock, replays two saved responses, records
10 activations, builds one dictionary at a time, and computes all five methods.

It passes only when:

- dictionary scores reproduce ordinary J-Lens top-50 rankings;
- our unmasked GP support matches TransformerLens 3.8.1;
- emitted token IDs never enter the masked primary supports;
- every decomposition is finite, nonempty, and uses at most 16 atoms;
- peak allocated VRAM stays below 75 GiB;
- all expected rows were saved.

In [ ]:
RUN_GPU_SMOKE = False
smoke_pointer_path = PROJECT_ROOT / "results" / "latest_rock_jspace_smoke_run.json"

if smoke_pointer_path.exists():
    smoke_pointer = load_json(smoke_pointer_path)
    smoke_completion = load_json(
        PROJECT_ROOT / "results" / smoke_pointer["run_id"] / "jspace_completion.json"
    )
    display(smoke_completion)
else:
    assert RUN_GPU_SMOKE, "Wait for Rock n=100, then explicitly enable the two-response smoke."
    subprocess.run(
        [PYTHON, "scripts/run_rock_jspace.py", "smoke"],
        cwd=PROJECT_ROOT,
        check=True,
    )
    smoke_pointer = load_json(smoke_pointer_path)
    smoke_completion = load_json(
        PROJECT_ROOT / "results" / smoke_pointer["run_id"] / "jspace_completion.json"
    )
    display(smoke_completion)

### Inspect the smoke rather than accepting only a green flag

This shows the two parity records and one paired raw example. The sparse
support is deliberately uncleaned: punctuation or unusual tokens remain
visible rather than being removed after the fact.

In [ ]:
smoke_dir = PROJECT_ROOT / "results" / smoke_pointer["run_id"]
parity = load_json(smoke_dir / "dictionary_parity.json")
ordinary_smoke = pd.read_parquet(smoke_dir / "ordinary_readouts.parquet")
jspace_smoke = pd.read_parquet(smoke_dir / "jspace_readouts.parquet")
display(pd.DataFrame(parity["records"]))
display(
    ordinary_smoke[[
        "prompt_id", "anchor", "method", "target_rank",
        "target_candidate_rank", "top10_json",
    ]].head(15)
)
display(
    jspace_smoke[[
        "prompt_id", "anchor", "method", "target_in_support",
        "target_contribution_share", "jspace_projection_fraction", "support_json",
    ]].head(10)
)

## Stage B — resumable 100 × 5 sweep

This stage is intentionally disabled. Enable it only after reviewing the smoke.
It creates a new immutable run and writes one activation/readout cell per prompt,
so rerunning the same `FULL_RUN_ID` resumes instead of overwriting completed work.

Model-facing work must run under `tmux`, not as a fragile long notebook cell.
The launch cell merely starts the checked-in runner and prints the log command.

In [ ]:
START_FULL_RUN = False
stamp = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
FULL_RUN_ID = f"run_{stamp}_{config['run_name']}_full"
TMUX_SESSION = "rock-jspace-full"
LOG_PATH = PROJECT_ROOT / "logs" / f"{FULL_RUN_ID}.log"
FULL_COMMAND = [
    PYTHON,
    "scripts/run_rock_jspace.py",
    "full",
    "--run-id",
    FULL_RUN_ID,
]
print("run id:", FULL_RUN_ID)
print("command:", shlex.join(FULL_COMMAND))
print("log:", LOG_PATH)

if START_FULL_RUN:
    assert smoke_completion["status"] == "passed"
    LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    shell_command = f"{shlex.join(FULL_COMMAND)} 2>&1 | tee {shlex.quote(str(LOG_PATH))}"
    subprocess.run(
        ["tmux", "new-session", "-d", "-s", TMUX_SESSION, shell_command],
        cwd=PROJECT_ROOT,
        check=True,
    )
    print("started; monitor with:", f"tmux attach -t {TMUX_SESSION}")

## Monitor or resume

Do not create a second run when a full run already exists. Reuse its exact
`FULL_RUN_ID`; completed prompt cells are skipped. The final pointer is written
only after all integrity gates pass.

In [ ]:
full_pointer_path = PROJECT_ROOT / "results" / "latest_rock_jspace_run.json"
if full_pointer_path.exists():
    full_pointer = load_json(full_pointer_path)
    full_completion_path = (
        PROJECT_ROOT / "results" / full_pointer["run_id"] / "jspace_completion.json"
    )
    display(load_json(full_completion_path))
else:
    candidate_manifest = PROJECT_ROOT / "results" / FULL_RUN_ID / "manifest.json"
    if candidate_manifest.exists():
        display(load_json(candidate_manifest))
    else:
        print("No full run started. This is the intended hand-off state after smoke.")

## Output contract for notebook 11

A passing full run contains:

- `activation_index.parquet` — exact prompt/anchor/position provenance;
- `ordinary_readouts.parquet` — Logit, public J-Lens, Rock J-Lens;
- `jspace_readouts.parquet` — public and Rock GP `k=16`, emitted-ID masked;
- `jspace_robustness.parquet` — GP `k=8/25` and clearly labelled unmasked NNOMP;
- `dictionary_parity.json` — normalization and TransformerLens parity gates;
- `jspace_completion.json` — counts, wall time, peak VRAM, and all gates.

Raw activations and per-prompt cells remain in ignored artifact directories;
aggregated tables remain in the immutable run directory.